# 02 — Limpieza y Transformación INE

**TFG: Spain's Migratory Flow**  
**Input:** 4 CSVs crudos del INE (nivel municipio/distrito/sección)  
**Output:** `output/clean/ine_provincia_anio.csv` — dataset limpio a nivel provincia × año

### Decisiones de diseño (documentadas en `01_ine.ipynb`)
| Decisión | Elección | Motivo |
|----------|----------|--------|
| Nivel geográfico | **Municipio** → agregado a **Provincia** | TFG usa provincia como unidad |
| Método agregación | **Mediana** | Robusta a outliers (municipios pequeños extremos) |
| Separador decimal | Normalizar a **punto (.)** | Mixto en fuente: `1_renta` usa `.`, resto usa `,` |
| Valores faltantes | `NaN` (ya capturados en lectura) | INE usa `..` = dato no disponible |
| Imputación NaN | **Sin imputar** — columna `_flag` indica ausencia | Preservar integridad |
| Clave join | `cod_provincia` + `year` | Compatible con EVR y otros dominios |

In [5]:
# ── Librerías ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# ── Estilo visual ──────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# ── Rutas (auto-discovery: sube hasta encontrar /data) ────────────────────────
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR   = PROJECT_ROOT / 'data' / 'renta_ine'
OUTPUT_DIR = PROJECT_ROOT / 'output' / 'clean'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATA_DIR     :', DATA_DIR)
print('OUTPUT_DIR   :', OUTPUT_DIR)
print('DATA existe  :', DATA_DIR.exists())

# ── Ficheros ──────────────────────────────────────────────────────────────────
FILES = {
    'renta':               DATA_DIR / '1.Indicadores de renta media y mediana.csv',
    'fuente_ing':          DATA_DIR / '2.Distribución por fuente de ingresos.csv',
    'umbrales_edad':       DATA_DIR / '4.Porcentaje de población con ingresos por unidad de consumo por debajo de determinados umbrales fijos por sexo y tramos de edad.csv',
    'umbrales_nacionalid': DATA_DIR / '5.Porcentaje de población con ingresos por unidad de consumo por debajo de determinados umbrales fijos por sexo y nacionalidad.csv',
    'gini':                DATA_DIR / '9.Índice de Gini y Distribución de la renta P80P20.csv',
    'demografico':         DATA_DIR / '10.Indicadores demográficos.csv',
}

# ── Constantes ────────────────────────────────────────────────────────────────
ENCODING  = 'utf-8-sig'
SEPARATOR = ';'
NA_VALUES = ['..', '']

# ── Mapa código INE → nombre provincia ───────────────────────────────────────
COD_PROVINCIA = {
    '01':'Álava',        '02':'Albacete',     '03':'Alicante',    '04':'Almería',
    '05':'Ávila',        '06':'Badajoz',      '07':'Baleares',    '08':'Barcelona',
    '09':'Burgos',       '10':'Cáceres',      '11':'Cádiz',       '12':'Castellón',
    '13':'Ciudad Real',  '14':'Córdoba',      '15':'A Coruña',    '16':'Cuenca',
    '17':'Girona',       '18':'Granada',      '19':'Guadalajara', '20':'Gipuzkoa',
    '21':'Huelva',       '22':'Huesca',       '23':'Jaén',        '24':'León',
    '25':'Lleida',       '26':'La Rioja',     '27':'Lugo',        '28':'Madrid',
    '29':'Málaga',       '30':'Murcia',       '31':'Navarra',     '32':'Ourense',
    '33':'Asturias',     '34':'Palencia',     '35':'Las Palmas',  '36':'Pontevedra',
    '37':'Salamanca',    '38':'S.C. Tenerife','39':'Cantabria',   '40':'Segovia',
    '41':'Sevilla',      '42':'Soria',        '43':'Tarragona',   '44':'Teruel',
    '45':'Toledo',       '46':'Valencia',     '47':'Valladolid',  '48':'Bizkaia',
    '49':'Zamora',       '50':'Zaragoza',     '51':'Ceuta',       '52':'Melilla',
}

# ── Verificación final ────────────────────────────────────────────────────────
print('\n🔍 Verificando ficheros:')
all_ok = True
for nombre, path in FILES.items():
    estado = '✅' if path.exists() else '❌ NO ENCONTRADO'
    if not path.exists(): all_ok = False
    print(f'  {estado}  {nombre:<20} → {path.name[:60]}')

print('\n✅ Configuración lista — todos los ficheros encontrados' if all_ok else
      '\n❌ Revisa los ficheros marcados arriba antes de continuar')

PROJECT_ROOT : /workspaces/TFG_Spain-s_Migratory_Flow
DATA_DIR     : /workspaces/TFG_Spain-s_Migratory_Flow/data/renta_ine
OUTPUT_DIR   : /workspaces/TFG_Spain-s_Migratory_Flow/output/clean
DATA existe  : True

🔍 Verificando ficheros:
  ✅  renta                → 1.Indicadores de renta media y mediana.csv
  ✅  fuente_ing           → 2.Distribución por fuente de ingresos.csv
  ✅  umbrales_edad        → 4.Porcentaje de población con ingresos por unidad de consumo
  ✅  umbrales_nacionalid  → 5.Porcentaje de población con ingresos por unidad de consumo
  ✅  gini                 → 9.Índice de Gini y Distribución de la renta P80P20.csv
  ✅  demografico          → 10.Indicadores demográficos.csv

✅ Configuración lista — todos los ficheros encontrados


---
## 1. Carga estandarizada

Una función única para los 4 archivos que:
- Lee con parámetros correctos (BOM, separador, na_values)
- Filtra a nivel municipio
- Normaliza el separador decimal
- Extrae código de provincia

In [6]:
import os

print('📦 Tamaño de cada fichero en disco:')
total = 0
for nombre, path in FILES.items():
    size_mb = path.stat().st_size / 1_048_576
    total += size_mb
    print(f'  {nombre:<20} → {size_mb:>7.1f} MB')

print(f'\n  TOTAL                  → {total:>7.1f} MB')
print(f'\n  RAM disponible estimada: usa solo ~{total*4:.0f} MB en pandas (×4 por dtype=str)')

📦 Tamaño de cada fichero en disco:
  renta                →   327.4 MB
  fuente_ing           →   278.7 MB
  umbrales_edad        →   327.7 MB
  umbrales_nacionalid  →   347.0 MB
  gini                 →   102.1 MB
  demografico          →   338.4 MB

  TOTAL                  →  1721.4 MB

  RAM disponible estimada: usa solo ~6886 MB en pandas (×4 por dtype=str)


In [ ]:
def cargar_y_filtrar(path: Path) -> pd.DataFrame:
    """
    Lee solo las 6 columnas necesarias del CSV del INE.
    La columna indicador siempre es df.columns[4] (confirmado en 01_ine.ipynb).
    """
    # ── Leer solo cabecera para saber el nombre exacto de col[4] ─────────────
    cols_todas = pd.read_csv(path, sep=SEPARATOR, encoding=ENCODING,
                             nrows=0, dtype=str).columns.tolist()
    col_ind = cols_todas[4]  # siempre posición 4 en todos los CSVs del INE

    # ── Leer solo las 6 columnas que necesitamos ──────────────────────────────
    df = pd.read_csv(
        path,
        sep=SEPARATOR,
        encoding=ENCODING,
        na_values=NA_VALUES,
        usecols=['Municipios', 'Distritos', 'Secciones', 'Periodo', 'Total', col_ind],
        dtype=str,
        low_memory=True
    )

    # ── Renombrar ─────────────────────────────────────────────────────────────
    df = df.rename(columns={
        col_ind:      'indicador',
        'Municipios': 'municipio',
        'Distritos':  'distrito',
        'Secciones':  'seccion',
        'Periodo':    'year',
        'Total':      'valor_raw',
    })

    # ── Filtrar solo municipio → elimina ~80% de filas ────────────────────────
    df = df[(df['distrito'].isna()) & (df['seccion'].isna())].copy()
    df.drop(columns=['distrito', 'seccion'], inplace=True)

    # ── Códigos geográficos ───────────────────────────────────────────────────
    df['cod_municipio'] = df['municipio'].str.extract(r'^(\d{5})')
    df['cod_provincia'] = df['municipio'].str.extract(r'^(\d{2})')
    df['nombre_prov']   = df['cod_provincia'].map(COD_PROVINCIA)

    # ── Normalizar decimal y convertir ───────────────────────────────────────
    df['valor'] = (
        df['valor_raw'].str.strip()
        .str.replace(',', '.', regex=False)
        .pipe(pd.to_numeric, errors='coerce')
    )
    df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
    df.drop(columns=['valor_raw'], inplace=True)

    return df[['municipio', 'cod_municipio', 'cod_provincia', 'nombre_prov',
               'year', 'indicador', 'valor']]


# ── Cargar los 6 datasets → se guardan en dfs_clean ──────────────────────────
print('⏳ Cargando datasets...\n')
dfs_clean = {}
for nombre, path in FILES.items():
    try:
        print(f'  ⏳ {nombre:<20}', end=' ', flush=True)
        dfs_clean[nombre] = cargar_y_filtrar(path)
        n     = len(dfs_clean[nombre])
        nulos = dfs_clean[nombre]['valor'].isna().sum()
        mem   = dfs_clean[nombre].memory_usage(deep=True).sum() / 1_048_576
        print(f'✅  {n:>8,} filas | NaN: {nulos:,} ({nulos/n*100:.1f}%) | {mem:.0f} MB')
    except Exception as e:
        print(f'❌  ERROR: {e}')

print('\n✅ Carga completada — variable: dfs_clean')

⏳ Cargando datasets...

  ⏳ renta                ❌  ERROR: "['indicador'] not in index"
  ⏳ fuente_ing           ❌  ERROR: "['indicador'] not in index"
  ⏳ umbrales_edad        ✅   234,252 filas | NaN: 164,727 (70.3%) | 25 MB
  ⏳ umbrales_nacionalid  ✅   288,360 filas | NaN: 258,588 (89.7%) | 30 MB
  ⏳ gini                 ❌  ERROR: "['indicador'] not in index"
  ⏳ demografico          ❌  ERROR: "['indicador'] not in index"

✅ Carga completada — variable: dfs_clean


: 

---
## 2. Validación de la conversión numérica

Comprobamos que la normalización decimal no ha generado pérdidas inesperadas.

In [ ]:
print('📋 Validación de conversión numérica:')
print(f'{"Fuente":<15} {"Filas":<10} {"NaN antes":>10} {"NaN después":>12} {"Diferencia":>12}')
print('-' * 62)

for nombre, path in FILES.items():
    # Contar NaN originales (antes de convertir)
    df_raw = pd.read_csv(path, sep=SEPARATOR, encoding=ENCODING,
                         na_values=NA_VALUES, dtype=str, low_memory=False)
    df_raw_mun = df_raw[(df_raw['Distritos'].isna()) & (df_raw['Secciones'].isna())]
    nan_antes = df_raw_mun['Total'].isna().sum()
    
    df_limpio = dfs_clean[nombre]
    nan_despues = df_limpio['valor'].isna().sum()
    diff = nan_despues - nan_antes
    
    estado = '✅' if diff == 0 else f'⚠️ +{diff}'
    print(f'{nombre:<15} {len(df_limpio):<10,} {nan_antes:>10,} {nan_despues:>12,} {estado:>12}')

print('\n💡 Si Diferencia > 0: hay valores no numéricos inesperados → revisar.')

📋 Validación de conversión numérica:
Fuente          Filas       NaN antes  NaN después   Diferencia
--------------------------------------------------------------


---
## 3. Detección y tratamiento de outliers

Usamos el método **IQR (rango intercuartílico)** por indicador y año.  
Estrategia: **no eliminar**, sino marcar con flag y Winsorizar (clamp a percentil 1-99).

> **¿Por qué no eliminar?** Los municipios extremos son reales (Pozuelo de Alarcón tiene renta muy alta;  
> municipios rurales despoblados tienen rentas muy bajas). Eliminarlos sesgaría el análisis.

In [ ]:
def detectar_outliers_iqr(df: pd.DataFrame, k: float = 3.0) -> pd.DataFrame:
    """
    Detecta outliers por grupo (indicador × año) usando IQR * k.
    Añade columnas 'is_outlier' y 'valor_wins' (winsorizado a p1-p99).
    k=3 es más permisivo que el estándar 1.5 (apropiado para datos geográficos).
    """
    df = df.copy()
    df['is_outlier']  = False
    df['valor_wins']  = df['valor']
    
    for (ind, yr), grupo in df.groupby(['indicador', 'year']):
        vals = grupo['valor'].dropna()
        if len(vals) < 10:
            continue
        
        q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
        iqr    = q3 - q1
        lower  = q1 - k * iqr
        upper  = q3 + k * iqr
        
        mask_out = (df['indicador'] == ind) & (df['year'] == yr)
        df.loc[mask_out, 'is_outlier'] = (
            (df.loc[mask_out, 'valor'] < lower) |
            (df.loc[mask_out, 'valor'] > upper)
        )
        # Winsorizar (clip a p1-p99)
        p01, p99 = vals.quantile(0.01), vals.quantile(0.99)
        df.loc[mask_out, 'valor_wins'] = df.loc[mask_out, 'valor'].clip(p01, p99)
    
    return df


# Aplicar a los 4 datasets
print('🔍 Detección de outliers (IQR × 3.0) por indicador y año:')
for nombre in dfs_clean:
    dfs_clean[nombre] = detectar_outliers_iqr(dfs_clean[nombre])
    n_out = dfs_clean[nombre]['is_outlier'].sum()
    n_tot = len(dfs_clean[nombre])
    print(f'  {nombre:<15}: {n_out:,} outliers ({n_out/n_tot*100:.2f}%)')

print('\n✅ Columnas añadidas: is_outlier, valor_wins')

In [ ]:
# Visualizar outliers por indicador (ejemplo: renta)
df_rent = dfs_clean['renta']
indicadores = df_rent['indicador'].unique()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, ind in enumerate(indicadores):
    if i >= len(axes): break
    sub = df_rent[(df_rent['indicador'] == ind) & (df_rent['year'] == 2023)]
    
    ax = axes[i]
    normales  = sub[~sub['is_outlier']]['valor'].dropna()
    outliers  = sub[sub['is_outlier']]['valor'].dropna()
    
    ax.hist(normales,  bins=40, color='#1976D2', alpha=0.8, label='Normal')
    ax.hist(outliers,  bins=10, color='#E53935', alpha=0.9, label=f'Outlier ({len(outliers)})')
    ax.set_title(ind, fontsize=9)
    ax.legend(fontsize=8)
    ax.set_xlabel('€')

for j in range(len(indicadores), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución con outliers marcados — Renta (2023, nivel municipio)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. Pivoting: formato largo → ancho por indicador

Pasamos de `(municipio, year, indicador, valor)` a `(municipio, year, ind1, ind2, ...)`  
para poder agregar después a nivel provincia.

In [ ]:
def limpiar_nombre_columna(s: str) -> str:
    """
    Convierte el nombre de un indicador INE a un nombre de columna limpio:
    minúsculas, sin tildes, espacios → guión bajo.
    """
    traducciones = str.maketrans('áéíóúüñÁÉÍÓÚÜÑ', 'aeiouunAEIOUUN')
    return (
        s.lower()
        .translate(traducciones)
        .replace(' ', '_')
        .replace('%', 'pct')
        .replace('/', '_')
        .replace('-', '_')
        .replace('(', '').replace(')', '')
        .replace(':', '')
        .replace('ó', 'o')
        .replace('  ', '_')
    )


def pivotar_dataset(df: pd.DataFrame, prefijo: str, usar_wins: bool = True) -> pd.DataFrame:
    """
    Pivota un dataset de formato largo a ancho.
    Usa valor_wins (winsorizado) si usar_wins=True, si no usar valor original.
    Añade columna de conteo de municipios por provincia-año-indicador.
    """
    col_valor = 'valor_wins' if usar_wins else 'valor'
    
    # Limpiar nombres de indicadores
    df = df.copy()
    df['indicador_col'] = prefijo + '__' + df['indicador'].apply(limpiar_nombre_columna)
    
    # Pivot: una fila por municipio × año
    pivot = df.pivot_table(
        index=['cod_municipio', 'municipio', 'cod_provincia', 'nombre_prov', 'year'],
        columns='indicador_col',
        values=col_valor,
        aggfunc='first'   # cada combinación municipio×año×indicador tiene 1 valor
    ).reset_index()
    
    pivot.columns.name = None  # limpiar nombre del eje de columnas
    return pivot


# Ejecutar el pivot para cada fuente
pivots = {}
prefijos = {
    'renta':       'renta',
    'fuente_ing':  'ingreso',
    'gini':        'desig',
    'demografico': 'demog',
}

for nombre, prefijo in prefijos.items():
    pivots[nombre] = pivotar_dataset(dfs_clean[nombre], prefijo)
    print(f'  ✓ {nombre:<15}: {pivots[nombre].shape}  columnas: {list(pivots[nombre].columns)}')

---
## 5. Merge horizontal: unir los 4 pivots por municipio × año

Join `LEFT` en cascada. La clave es `cod_municipio + year`.

In [ ]:
KEYS = ['cod_municipio', 'municipio', 'cod_provincia', 'nombre_prov', 'year']

# Empezar con renta como base (máxima cobertura)
df_merged = pivots['renta'].copy()
for nombre in ['fuente_ing', 'gini', 'demografico']:
    df_right = pivots[nombre]
    df_merged = df_merged.merge(df_right, on=KEYS, how='left', suffixes=('', f'_{nombre}'))

print(f'✅ Dataset a nivel municipio × año: {df_merged.shape}')
print(f'   Columnas: {list(df_merged.columns)}')
print(f'\n📊 Sample (5 filas):')
print(df_merged.head(5).to_string(max_colwidth=30))

---
## 6. Agregación a nivel PROVINCIA

Para el TFG la unidad de análisis es **provincia × año**.  
Usamos la **mediana** de todos los municipios de cada provincia.

In [ ]:
# Columnas de indicadores (todo excepto las claves)
cols_indicadores = [c for c in df_merged.columns if c not in KEYS]

# Mediana de municipios por provincia × año
df_prov = (
    df_merged
    .groupby(['cod_provincia', 'nombre_prov', 'year'])[cols_indicadores]
    .median()
    .reset_index()
)

# Añadir conteo de municipios con dato válido (renta neta media como proxy)
col_renta_ref = [c for c in cols_indicadores if 'renta_neta_media_por_persona' in c]
if col_renta_ref:
    n_mun_validos = (
        df_merged.groupby(['cod_provincia', 'year'])[col_renta_ref[0]]
        .count()
        .reset_index(name='n_municipios_con_dato')
    )
    df_prov = df_prov.merge(n_mun_validos, on=['cod_provincia', 'year'], how='left')

print(f'✅ Dataset provincia × año: {df_prov.shape}')
print(f'   Provincias: {df_prov["cod_provincia"].nunique()}')
print(f'   Años: {sorted(df_prov["year"].dropna().unique())}')
print(f'\n📊 Sample:')
print(df_prov.head(10).to_string(max_colwidth=25))

---
## 7. Verificación de cobertura: heatmap de completitud

¿Qué porcentaje de valores está disponible para cada indicador × año a nivel provincia?

In [ ]:
# Calcular % completitud por columna × año
df_prov_2 = df_prov.copy()
cols_plot  = [c for c in df_prov_2.columns if c not in ['cod_provincia', 'nombre_prov', 'year', 'n_municipios_con_dato']]

completitud = (
    df_prov_2.groupby('year')[cols_plot]
    .apply(lambda g: g.notna().mean() * 100)
    .T   # filas = indicadores, columnas = años
)

fig, ax = plt.subplots(figsize=(14, max(6, len(completitud) * 0.45)))
sns.heatmap(
    completitud, annot=True, fmt='.0f', cmap='RdYlGn',
    vmin=0, vmax=100, linewidths=0.3,
    cbar_kws={'label': '% provincias con dato'},
    ax=ax
)
ax.set_title('Completitud por indicador y año — nivel provincia', pad=12)
ax.set_xlabel('Año')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

# Indicadores con < 80% completitud en algún año
baja_cobertura = completitud[completitud.min(axis=1) < 80]
if not baja_cobertura.empty:
    print('\n⚠️  Indicadores con completitud < 80% en algún año:')
    print(baja_cobertura.to_string())
else:
    print('\n✅ Todos los indicadores superan 80% de completitud en todos los años')

---
## 8. Análisis de consistencia temporal

Verificamos que las series temporales no tienen saltos anómalos entre años consecutivos.

In [ ]:
# Calcular variación interanual (%) por provincia e indicador
def variacion_interanual(df: pd.DataFrame, col: str) -> pd.DataFrame:
    df_sorted = df.sort_values(['cod_provincia', 'year'])
    df_sorted['delta_pct'] = df_sorted.groupby('cod_provincia')[col].pct_change() * 100
    return df_sorted[['cod_provincia', 'nombre_prov', 'year', col, 'delta_pct']].dropna()


# Usar renta neta media como ejemplo
col_ref = [c for c in df_prov.columns if 'renta_neta_media_por_persona' in c]
if col_ref:
    col_ref = col_ref[0]
    df_var = variacion_interanual(df_prov, col_ref)
    
    # Detectar variaciones > 20% (potencialmente erróneas)
    anomalias = df_var[df_var['delta_pct'].abs() > 20]
    
    print(f'📊 Variación interanual — {col_ref}')
    print(f'   Media: {df_var["delta_pct"].mean():.1f}%')
    print(f'   P5-P95: [{df_var["delta_pct"].quantile(0.05):.1f}%, {df_var["delta_pct"].quantile(0.95):.1f}%]')
    print(f'   Anomalías (>±20%): {len(anomalias)}')
    if not anomalias.empty:
        print('\n   ⚠️  Detalle anomalías:')
        print(anomalias.to_string(index=False))
    
    # Boxplot variaciones por año
    fig, ax = plt.subplots(figsize=(12, 4))
    df_var.boxplot(column='delta_pct', by='year', ax=ax)
    ax.axhline(0, color='red', linestyle='--', alpha=0.5)
    ax.set_title(f'Variación interanual (%) — {col_ref}')
    ax.set_xlabel('Año')
    ax.set_ylabel('Δ% respecto año anterior')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

---
## 9. Export: guardado del dataset limpio

Guardamos 2 archivos:
- `ine_provincia_anio.csv` — dataset limpio a nivel provincia × año (para el merge con EVR)
- `ine_municipio_anio.csv` — dataset municipio × año (para análisis granular opcional)

In [ ]:
# ── Export provincia × año ─────────────────────────────────────────────────────
path_prov = OUTPUT_DIR / 'ine_provincia_anio.csv'
df_prov.to_csv(path_prov, index=False, encoding='utf-8')
print(f'✅ Guardado: {path_prov}')
print(f'   Shape: {df_prov.shape}')
print(f'   Columnas: {list(df_prov.columns)}')

# ── Export municipio × año ─────────────────────────────────────────────────────
path_mun = OUTPUT_DIR / 'ine_municipio_anio.csv'
df_merged.to_csv(path_mun, index=False, encoding='utf-8')
print(f'\n✅ Guardado: {path_mun}')
print(f'   Shape: {df_merged.shape}')

# ── Verificación de integridad ──────────────────────────────────────────────────
df_check = pd.read_csv(path_prov)
assert df_check.shape == df_prov.shape, '❌ Error: shape no coincide en reload'
print(f'\n✅ Integridad verificada: reload OK')

# ── Resumen del output ─────────────────────────────────────────────────────────
print('\n' + '='*65)
print('📋 RESUMEN DE LIMPIEZA')
print('='*65)
print(f'  Output principal : ine_provincia_anio.csv')
print(f'  Observaciones    : {len(df_prov):,} (52 prov × 9 años = {52*9})')
print(f'  Variables        : {len(df_prov.columns)} columnas')
print(f'  Cobertura años   : {sorted(df_prov["year"].dropna().unique())}')
print(f'  Provincias       : {df_prov["cod_provincia"].nunique()}/52')
print(f'  NaN totales      : {df_prov.isna().sum().sum():,}')
print(f'  Completitud      : {(1 - df_prov.isna().mean().mean())*100:.1f}%')
print('\n  → Listo para notebook 03_ine.ipynb (visualizaciones finales)')

---
## 10. Diccionario de variables del output

Documento las columnas del dataset limpio para el resto del pipeline.

In [ ]:
diccionario = []
for col in df_prov.columns:
    dtype   = str(df_prov[col].dtype)
    n_nulos = df_prov[col].isna().sum()
    ejemplo = df_prov[col].dropna().iloc[0] if df_prov[col].notna().any() else 'N/A'
    
    # Descripción automática basada en nombre
    if col == 'cod_provincia':  desc = 'Código INE de provincia (2 dígitos)'
    elif col == 'nombre_prov':  desc = 'Nombre de la provincia'
    elif col == 'year':         desc = 'Año de observación'
    elif col == 'n_municipios_con_dato': desc = 'Nº de municipios con dato válido en la provincia'
    elif 'renta_neta' in col:   desc = 'Renta neta media (€) — mediana municipal'
    elif 'renta_bruta' in col:  desc = 'Renta bruta media (€) — mediana municipal'
    elif 'mediana' in col:      desc = 'Mediana de renta por unidad de consumo'
    elif 'salario' in col:      desc = '% hogares con salario como fuente principal'
    elif 'pension' in col:      desc = '% hogares con pensiones como fuente principal'
    elif 'desempleo' in col:    desc = '% hogares con prestación por desempleo'
    elif 'gini' in col:         desc = 'Índice de Gini (desigualdad, 0-100)'
    elif 'p80' in col or 'p20' in col: desc = 'Ratio P80/P20 (brecha ricos/pobres)'
    elif 'edad_media' in col:   desc = 'Edad media de la población'
    elif '65' in col:           desc = '% población ≥65 años'
    elif '18' in col:           desc = '% población <18 años'
    elif 'hogar' in col:        desc = 'Tamaño medio del hogar o % hogares unipersonales'
    elif 'poblacion' in col or 'población' in col: desc = 'Población total o % española'
    else:                       desc = col
    
    diccionario.append({
        'Columna': col,
        'Tipo': dtype,
        'NaN': n_nulos,
        'Ejemplo': str(ejemplo)[:30],
        'Descripción': desc
    })

df_dict = pd.DataFrame(diccionario)
print('📖 DICCIONARIO DE VARIABLES — ine_provincia_anio.csv')
print(df_dict.to_string(index=False))

# Guardar diccionario
df_dict.to_csv(OUTPUT_DIR / 'ine_diccionario_variables.csv', index=False, encoding='utf-8')
print('\n✅ Diccionario guardado en output/clean/ine_diccionario_variables.csv')